# TCC 2 - Predição de Safra

## Carregamento e Preparação Inicial

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Carregando os arquivos
print("Carregando os datasets...")
df_clima = pd.read_csv('../Data/agroclimatologia_completa_enso.csv')
df_prod = pd.read_csv('../Data/produtividade_soja_completa.csv')

# Convertendo a coluna de data de string para o formato datetime
df_clima['data'] = pd.to_datetime(df_clima['data'])


print("Transformando dados de produtividade (Wide -> Long)...")
# Lista com os anos que estão como colunas
anos_safra = [str(ano) for ano in range(2004, 2025)]

# Aplicando o pd.melt para empilhar os anos
df_prod_long = pd.melt(
    df_prod,
    id_vars=['codigo_ibge', 'name'], # Colunas que mantidas fixas
    value_vars=anos_safra,           # Colunas que serão derretidas
    var_name='ano_safra',            # Nome da nova coluna de anos
    value_name='produtividade_kg_ha' # Nome da variável alvo
)

df_prod_long['ano_safra'] = df_prod_long['ano_safra'].astype(int)
df_prod_long = df_prod_long.dropna(subset=['produtividade_kg_ha'])

# Resultados
print(f"Base de clima carregada com {df_clima.shape[0]} linhas e {df_clima.shape[1]} colunas.")
print(f"Base de produtividade alinhada com {df_prod_long.shape[0]} registros válidos.")
display(df_prod_long.head())

# Se quiser salvar depois
# df_prod_long.to_csv('produtividade_long_derretida.csv', index=False)

Carregando os datasets...
Transformando dados de produtividade (Wide -> Long)...
Base de clima carregada com 2931949 linhas e 42 colunas.
Base de produtividade alinhada com 7329 registros válidos.


,codigo_ibge,name,ano_safra,produtividade_kg_ha
0,4100103,Abatiá,2004,2600.0
1,4100459,Altamira do Paraná,2004,3100.0
2,4100608,Alto Paraná,2004,1487.0
3,4100707,Alto Piquiri,2004,2300.0
4,4100509,Altônia,2004,1050.0


## Higienização

In [ ]:
print("Iniciando varredura para corrigir colunas numéricas corrompidas...")

# Protegendo as colunas que são legitimamente textos ou identificadores
colunas_texto_seguras = ['data', 'fdata', 'codigo_ibge', 'latitude', 'longitude', 'name', 'fenomeno_enso']

# Forçando a conversão de todo o resto para número decimal (float)
for col in df_clima.columns:
    if col not in colunas_texto_seguras:
        # Transformando textos sujos em NaN, salvando a coluna
        df_clima[col] = pd.to_numeric(df_clima[col], errors='coerce')

print("Varredura concluída! Todas as variáveis climáticas agora são estritamente matemáticas.")

Iniciando varredura para corrigir colunas numéricas corrompidas...
Varredura concluída! Todas as variáveis climáticas agora são estritamente matemáticas.


## Janelamento Temporal

In [15]:
import pandas as pd
import numpy as np

# Função para calcular a amplitude (Max - Min)
def amplitude(x):
    return x.max() - x.min()

# Renomeando a função para que a coluna no Pandas fique com o nome 'amplitude'
amplitude.__name__ = 'amplitude'

def criar_janela_climatica(df_clima, ano_safra, mes_previsao, janela_meses):
    """
    Filtra os dados climáticos para uma janela de X meses antes do mês de previsão
    e extrai as estatísticas descritivas complexas e o índice ENSO (ONI).
    """
    # Ajustando o Ano (Safra ocorre entre Setembro e Maio)
    ano_civil = ano_safra if mes_previsao <= 5 else ano_safra - 1
    
    # Data exata em que a previsão será feita (ex: 1º de Janeiro)
    data_ref = pd.to_datetime(f"{ano_civil}-{mes_previsao:02d}-01")
    
    # Data de início da janela PASSADA (ex: 2 meses atrás)
    data_inicio = data_ref - pd.DateOffset(months=janela_meses)
    
    # Filtrando o dataset de clima estritamente para a janela de tempo climática
    mascara = (df_clima['data'] >= data_inicio) & (df_clima['data'] < data_ref)
    df_janela = df_clima[mascara]
    
    # Se por algum motivo não houver dados na janela climática, retorna vazio para evitar erros
    if df_janela.empty:
        return pd.DataFrame()
    
    # Lista de estatísticas que usaremos
    # std = desvio padrão, skew = assimetria, kurt = curtose
    stats_base = ['mean', 'median', 'min', 'max', amplitude, 'std', 'skew', 'kurt']

    colunas_ignorar = ['data', 'codigo_ibge', 'latitude', 'longitude', 'fenomeno_enso', 'fenomeno_enso_numerico', 'oni']
    
    variaveis_climaticas = [col for col in df_clima.columns if col not in colunas_ignorar]
    
    agg_dict = {}
    for var in variaveis_climaticas:
        if var == 'PRECTOTCORR':
            # A precipitação recebe a soma (acumulado) + as estatísticas base
            agg_dict[var] = ['sum'] + stats_base
        else:
            # Todas as outras recebem apenas as estatísticas base
            agg_dict[var] = stats_base
            
    # Agrupamento por município usando o dicionário dinâmico
    df_agg = df_janela.groupby('codigo_ibge').agg(agg_dict)
    
    # Achatando as colunas
    df_agg.columns = [f"{col[0]}_{col[1]}" for col in df_agg.columns]
    
    # Tratamento do Índice ENSO (ONI)
    df_enso = df_janela.groupby('codigo_ibge').agg({
        'oni': 'last',                      
        'fenomeno_enso': 'last',            
        'fenomeno_enso_numerico': 'last'    
    })
    
    # Juntar os atributos climáticos com o ENSO
    df_final = df_agg.join(df_enso).reset_index()
    
    # Anexar informações de contexto
    df_final['ano_safra'] = ano_safra
    df_final['mes_previsao'] = mes_previsao
    df_final['janela_meses'] = janela_meses
    
    return df_final

## Loop de Janelamento e Cruzamento Final

In [ ]:
# Definindo as regras da metodologia
meses_previsao_safra = [11, 1, 3, 5] # Novembro, Janeiro, Março, Maio
tamanhos_janela_climatica = [1, 2, 3] # Janelas climáticas de 1, 2 e 3 meses passados
anos_safra = df_prod_long['ano_safra'].unique() # Todos os anos que temos na base de produção

lista_datasets_janelas = []

# Loop por todas as combinações
for ano in anos_safra:
    for mes in meses_previsao_safra:
        for janela in tamanhos_janela_climatica:
            # Gerandondo a janela climática
            df_temp = criar_janela_climatica(df_clima, ano, mes, janela)
            
            if not df_temp.empty:
                lista_datasets_janelas.append(df_temp)

# Concatenando todos os blocos em um único DataFrame de clima achatado
df_clima_completo = pd.concat(lista_datasets_janelas, ignore_index=True)

# Cruzamento Final (Merge) com a tabela de Produtividade (Target)
df_master = pd.merge(
    df_clima_completo, 
    df_prod_long, 
    on=['codigo_ibge', 'ano_safra'], 
    how='inner'
)

print(f"Dataset final (df_master) construído com sucesso!")
print(f"Total de amostras prontas para o Machine Learning: {df_master.shape[0]} linhas.")
display(df_master.head())

Dataset final (df_master) construído com sucesso!
Total de amostras prontas para o Machine Learning: 87948 linhas.


,codigo_ibge,fdata_mean,fdata_median,fdata_min,fdata_max,fdata_amplitude,fdata_std,fdata_skew,fdata_kurt,T2M_mean,...,ALLSKY_SFC_UV_INDEX_skew,ALLSKY_SFC_UV_INDEX_kurt,oni,fenomeno_enso,fenomeno_enso_numerico,ano_safra,mes_previsao,janela_meses,name,produtividade_kg_ha
0,4100103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.935161,...,-5.567685,30.999392,0.2,Neutro,0,2004,11,1,Abatiá,2600.0
1,4100459,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.538387,...,-3.728110,12.716813,0.2,Neutro,0,2004,11,1,Altamira do Paraná,3100.0
2,4100509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.056452,...,-5.567671,30.999284,0.2,Neutro,0,2004,11,1,Altônia,1050.0
3,4100608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.731613,...,-3.728117,12.716845,0.2,Neutro,0,2004,11,1,Alto Paraná,1487.0
4,4100707,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.056452,...,-5.567655,30.999165,0.2,Neutro,0,2004,11,1,Alto Piquiri,2300.0
